## LightGBM raw test-set predictions for bootstrapping/calibration (Olink -> SomaScan)

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("..."), "..")))
project_root = os.path.abspath(os.path.expanduser("..."))
sys.path.append(project_root)
os.chdir(project_root)

import pickle
import pandas as pd
import numpy as np

In [ ]:
MESA_Soma = pd.read_csv("Data/final_datasets/MESA_SOMA_5k_log2scaled_nodup_final.tsv.gz",
                        sep = "\t", compression = "gzip").drop(columns={"bmi"})
MESA_Olink = pd.read_csv("Data/final_datasets/MESA_Olink_3k_log2_nodup_rm10na_knn_final.tsv.gz",
                         sep = "\t", compression = "gzip")

overlap_IDs = np.unique(list(set(np.array(MESA_Olink["id"])).intersection(set(np.array(MESA_Soma["id"])))))

MESA_Olink  = MESA_Olink[MESA_Olink["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])
MESA_Soma   = MESA_Soma[MESA_Soma["id"].isin(overlap_IDs)].sort_values(by=["id"]).drop_duplicates(["id"])

MESA_Soma = MESA_Soma.set_index("id")
MESA_Olink = MESA_Olink.set_index("id")

In [ ]:
train_index = pickle.load(open("Data/helper_data/train_index.pkl", "rb"))
test_index = pickle.load(open("Data/helper_data/test_index.pkl", "rb"))

## OS-direction target list (SomaScan proteins)
protlist = MESA_Soma.columns
olink_cols = set(MESA_Olink.columns)

## Predict on MESA test set using saved LightGBM models

In [ ]:
%%time

all_test_values_os = []
X_test = MESA_Olink.loc[test_index]

for protein_name in protlist:
    model_path = f"Data/soma_trained_models/olink_soma/3kolink_5k_log2scaled_soma_lgb/{protein_name}_lgb.pkl"

    if not os.path.exists(model_path):
        print(f"skip {protein_name} - no saved LightGBM model found")
        continue

    reg = pickle.load(open(model_path, "rb"))
    print(f"predicting {protein_name}")

    y = MESA_Soma[protein_name]
    y_test = y.loc[test_index]
    y_pred = reg.predict(X_test)

    if protein_name in olink_cols:
        X_test_protein = MESA_Olink[protein_name].loc[test_index]
        x_test_vals = X_test_protein.values
    else:
        x_test_vals = np.nan

    per_protein_test_values = pd.DataFrame({
        "id": X_test.index,
        "protein": protein_name,
        "y_test": y_test.values,
        "y_pred": y_pred,
        "X_test_protein": x_test_vals
    })
    all_test_values_os.append(per_protein_test_values)

raw_test_values_os_df = pd.concat(all_test_values_os, ignore_index=True)

## Pivot to wide (n x p matrices) and save

In [ ]:
%%time

y_test_wide = raw_test_values_os_df.pivot(index="id", columns="protein", values="y_test")
y_pred_wide = raw_test_values_os_df.pivot(index="id", columns="protein", values="y_pred")
x_test_protein_wide = raw_test_values_os_df.pivot(index="id", columns="protein", values="X_test_protein")

print(y_test_wide.shape, y_pred_wide.shape, x_test_protein_wide.shape)

y_test_wide.reset_index().to_csv(
    "Results/mesa_test_bootstrap_inputs/lgb_mesa_3kolink_pred_5ksoma_y_test_wide.tsv.gz",
    sep="\t", index=False, compression="gzip"
)
y_pred_wide.reset_index().to_csv(
    "Results/mesa_test_bootstrap_inputs/lgb_mesa_3kolink_pred_5ksoma_y_pred_wide.tsv.gz",
    sep="\t", index=False, compression="gzip"
)
x_test_protein_wide.reset_index().to_csv(
    "Results/mesa_test_bootstrap_inputs/lgb_mesa_3kolink_pred_5ksoma_x_test_protein_wide.tsv.gz",
    sep="\t", index=False, compression="gzip"
)

## Evaluation performance

In [ ]:
%%time

from scipy.stats import pearsonr

results = []
present_proteins = raw_test_values_os_df["protein"].unique()

for protein_name in present_proteins:
    sub = raw_test_values_os_df[raw_test_values_os_df["protein"] == protein_name]
    y_test = sub["y_test"].values
    y_pred = sub["y_pred"].values
    x_test_protein = sub["X_test_protein"].values

    model_r, _ = pearsonr(y_test, y_pred)
    model_r2 = model_r ** 2

    if protein_name in olink_cols:
        y_train = MESA_Soma[protein_name].loc[train_index]
        X_train_protein = MESA_Olink[protein_name].loc[train_index]

        pretrain_r, _ = pearsonr(y_train, X_train_protein)
        pretrain_r2 = pretrain_r ** 2

        posttrain_r, _ = pearsonr(y_pred, x_test_protein)
        posttrain_r2 = posttrain_r ** 2

        testset_r, _ = pearsonr(y_test, x_test_protein)
        testset_r2 = testset_r ** 2
    else:
        pretrain_r = pretrain_r2 = posttrain_r = posttrain_r2 = testset_r = testset_r2 = np.nan

    results.append({
        "protein": protein_name,
        "has_soma": protein_name in olink_cols,
        "model_r": model_r, "model_r2": model_r2,
        "pretrain_r": pretrain_r, "pretrain_rsq": pretrain_r2,
        "posttrain_r": posttrain_r, "posttrain_rsq": posttrain_r2,
        "testset_r": testset_r, "testset_r2": testset_r2
    })
    
    print(f"evaluation done for {protein_name}")

In [ ]:
results_df = pd.DataFrame(results)
os.makedirs("Results/metrics", exist_ok=True)
results_df.to_csv("Results/metrics/LGB_3kolink_log2scaled_5ksoma_results.tsv", sep="\t", index=False)